# Basic Decision Tree from Scratch - Regression (Variance) - Optimization

In this notebook, we are going to optimize the Gini search algorithm we had. The optimization can be organized into:

- Optimization 1 : Sliding Windows
- Optimization 2 : Cumulative Sum
- Optimization 3 : Vectorization
- Optimization 4 : Reduce boundaries and remove feature values that are duplicates

## Dataset

Dataset: This is a makeup dataset that describe how much study hours a student put in and what exam score they can achieve.

Dataset 
| Study Hours  | Exam Score |
| ----- | ----- |
| 1    | 10    |
| 2    | 20    |
| 3    | 30    |
| 4    | 45    |
| 5    | 50   |
| 6    | 55   |
| 7    | 75   |
| 8    | 80   |
| 9    | 80   |
| 10    | 90   |
| 11    | 95   |
| 12    | 93   |
| 13    | 90   |
| 14    | 91   |
| 15    | 88   |

In [10]:
import pandas as pd
import numpy as np
import json
from typing import Optional

In [11]:
df = pd.DataFrame({
    'studied': [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15],
    'score':  [10,20,30,45,50,55,75,80,80,90,95,93,90,91,88]
})

In [12]:
df

,studied,score
0,1,10
1,2,20
2,3,30
3,4,45
4,5,50
5,6,55
6,7,75
7,8,80
8,9,80
9,10,90


## Variance Formula

### Variance Formula

For a node:

$$\text{Variance} = \frac{\sum (y_i - \bar{y})^2}{n}$$


Where:

- $y_i$ = target values
- $\bar{y}$ = mean of target values


**Intuition**

| Case                    | Variance      |
| ----------------------- | ------------- |
| target tightly grouped  | low variance  |
| target widely scattered | high variance |

**A regression tree chooses splits that reduce variance after splitting**

---

### Weighted Variance (for a split)

When you split a node into LEFT and RIGHT:

$$ \text{Weighted Variance} = \frac{N_L}{N} \cdot Var(L) + \frac{N_R}{N} \cdot Var(R)$$

Where:

- $N_L$ = number of samples in left node
- $N_R$ = number of samples in right node
- $N = N_L + N_R$


**Intuition**

Weighted Variance means:

> "How much uncertainty remains after the split, considering how many samples fall into each side?"

### Variance split procedure:
- Sort feature values
- For each adjacent pair:
    - compute midpoint threshold
- For each threshold:
    - split data
    - compute weighted variance
- Choose best threshold

## Midpoint Optimization

Midpoint can also be optimized but this is low priority as we optimized further, we only computed selected midpoint instead of computing all midpoint.

In [13]:
# compute mid points between values of studied
mid_points = []
for i in range(len(df['studied']) - 1):
    mid_point = (df['studied'][i] + df['studied'][i+1]) / 2
    # [i] refer to the fist value and [i+1] refer to the second value and so on
    mid_points.append(float(mid_point))

print(mid_points)

[1.5, 2.5, 3.5, 4.5, 5.5, 6.5, 7.5, 8.5, 9.5, 10.5, 11.5, 12.5, 13.5, 14.5]


In [14]:
x = df['studied'].values
x

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15])

In [15]:
print(x[:-1])
print(x[1:])

[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14]
[ 2  3  4  5  6  7  8  9 10 11 12 13 14 15]


In [16]:
mid_points = (x[:-1] + x[1:]) / 2
mid_points

array([ 1.5,  2.5,  3.5,  4.5,  5.5,  6.5,  7.5,  8.5,  9.5, 10.5, 11.5,
       12.5, 13.5, 14.5])

## Current Variance Algorithm

The current variance search algorithm we use `for loop` to calculate all midpoint and variance. Then we select the midpoint with least weighted variance.

The current loop is resource intensive because for every midpoint, we split the features into 2 portion.

In [17]:
# weighted variance of left and right splits
def weighted_variance(df_left, df_right):
    n_left = len(df_left)
    n_right = len(df_right)
    n_total = n_left + n_right

    var_left = df_left['score'].var(ddof=0)
    var_right = df_right['score'].var(ddof=0)

    #print(f"Variance for left split: {var_left:.2f}")
    #print(f"Variance for right split: {var_right:.2f}")

    weighted_var = (n_left / n_total) * var_left + (n_right / n_total) * var_right

    #print(f"Weighted variance: {weighted_var:.2f}")
    
    return weighted_var

In [18]:
def best_var_split(df):
    best_variance = float('inf')
    best_split_value = None
    
    x = df['studied'].values  # best practice: NumPy

    for i in range(len(x) - 1):
        mid_point = (x[i] + x[i+1]) / 2
        df_left = df[df['studied'] <= mid_point]
        df_right = df[df['studied'] > mid_point]
        
        current_variance = weighted_variance(df_left, df_right)
        print(f"variance: {current_variance:.2f}, threshold: {mid_point}")
        
        if current_variance < best_variance:
            best_variance = current_variance
            best_split_value = mid_point
            
    return best_split_value, best_variance

In [19]:
best_split_value, best_variance = best_var_split(df)
print(f"Best split value: {best_split_value}, Best variance: {best_variance:.2f}")

variance: 552.91, threshold: 1.5
variance: 375.73, threshold: 2.5
variance: 245.91, threshold: 3.5
variance: 199.55, threshold: 4.5
variance: 160.81, threshold: 5.5
variance: 131.79, threshold: 6.5
variance: 212.62, threshold: 7.5
variance: 297.31, threshold: 8.5
variance: 360.20, threshold: 9.5
variance: 458.78, threshold: 10.5
variance: 562.08, threshold: 11.5
variance: 639.53, threshold: 12.5
variance: 693.98, threshold: 13.5
variance: 743.83, threshold: 14.5
Best split value: 6.5, Best variance: 131.79


In [20]:
def build_tree_regression(df):

    # ---- STOP 1: empty node ----
    if df.empty:
        return None

    # ---- STOP 2: too small to split ----
    if len(df) <= 1:
        print("Too small to split, one row left, returning mean score")
        return float(df['score'].mean())
    
    # --- STOP 3: no variance ----
    if df['score'].var(ddof=0) == 0:
        print("No variance in score, returning mean score")
        return float(df['score'].mean())

    # ---- compute median split ----
    best_split_value, best_variance = best_var_split(df)

    print(f"Best split value: {best_split_value}, Best variance: {best_variance:.2f}")

    # ---- split data ----
    left_subset = df[df['studied'] <= best_split_value]
    right_subset = df[df['studied'] > best_split_value]

    print(f"Left subset:\n{left_subset}\n")
    print(f"Right subset:\n{right_subset}\n")
    
    # ---- build node ----
    node = {
        'threshold_studied': float(best_split_value),
        'value': round(float(df['score'].mean()), 2),  # 👈 REGRESSION CHANGE
        'left': None,
        'right': None
    }

    # ---- recursion ----
    node['left'] = build_tree_regression(left_subset)
    node['right'] = build_tree_regression(right_subset)

    return node

In [21]:
reg = build_tree_regression(df)

variance: 552.91, threshold: 1.5
variance: 375.73, threshold: 2.5
variance: 245.91, threshold: 3.5
variance: 199.55, threshold: 4.5
variance: 160.81, threshold: 5.5
variance: 131.79, threshold: 6.5
variance: 212.62, threshold: 7.5
variance: 297.31, threshold: 8.5
variance: 360.20, threshold: 9.5
variance: 458.78, threshold: 10.5
variance: 562.08, threshold: 11.5
variance: 639.53, threshold: 12.5
variance: 693.98, threshold: 13.5
variance: 743.83, threshold: 14.5
Best split value: 6.5, Best variance: 131.79
Left subset:
   studied  score
0        1     10
1        2     20
2        3     30
3        4     45
4        5     50
5        6     55

Right subset:
    studied  score
6         7     75
7         8     80
8         9     80
9        10     90
10       11     95
11       12     93
12       13     90
13       14     91
14       15     88

variance: 141.67, threshold: 1.5
variance: 66.67, threshold: 2.5
variance: 41.67, threshold: 3.5
variance: 113.54, threshold: 4.5
variance: 186

## Optimization 1: Sliding Windows Optimization

Pseudo code:

- Sort feature
- Initialize LEFT counts to zero
- Initialize RIGHT counts to total class counts
- Initialized sum of y and sum of y square
- Move one sample at a time from RIGHT to LEFT
- Update counts
- Update sum of y and sum of y square
- Compute weighted variance
- Track best boundary
- Convert best boundary to midpoint threshold

**The key advantage about this algorithm is that instead of actually splitting the feature, we calculate the total number of positive and negative label by adding 1 or subtracting 1 based on the labels of feature value that we are examining as we slide across the feature value one by one. Thus this is call sliding windows.** 

### Variance Derivation

$$\text{Variance} = \frac{\sum (y_i - \bar{y})^2}{n}$$
$$\text{Variance} = \frac{\sum(y_i - \bar{y})(y_i - \bar{y})}{n}$$
$$\text{Variance} = \frac{\sum(y_i^2 - 2\bar{y}y_i +\bar{y}^2)}{n}$$
$$\text{Variance} = \frac{\sum{y_i^2} - \sum{2\bar{y}y_i} +\sum{\bar{y}^2}}{n}$$
$$\text{Variance} = \frac{\sum{y_i^2} - 2\bar{y}\sum{y_i} +\sum{\bar{y}^2}}{n}$$

$$\bar{y} = \frac{\sum{y_i}}{n} \rightarrow \sum{y_i} = \bar{y}n$$
$$\text{Substituting} \sum{y_i} = \bar{y}n$$

$$\text{Variance} = \frac{\sum{y_i^2} - 2\bar{y}\bar{y}n +\sum{\bar{y}^2}}{n}$$
$$\text{Variance} = \frac{\sum{y_i^2} - 2n\bar{y}^2 +\sum{\bar{y}^2}}{n}$$

$$\text{Since } \bar{y} \text{ is constant, therefore,} \sum{\bar{y}^2} = n\bar{y}^2$$
$$\text{Substituting } \sum{\bar{y}^2} = n\bar{y}^2$$
$$\text{Variance} = \frac{\sum{y_i^2} - 2n\bar{y}^2 +n\bar{y}^2}{n}$$


$$\text{Variance} = \frac{\sum{y_i^2} - n\bar{y}^2 }{n}$$
$$\text{Variance} = \frac{\sum{y_i^2}}{n} - \frac{n\bar{y}^2 }{n}$$
$$\text{Variance} = \frac{\sum{y_i^2}}{n} - \bar{y}^2$$

$$\text{Since }\bar{y} = \frac{\sum{y_i}}{n}$$
$$\text{Variance} = \frac{\sum{y_i^2}}{n} - \left(\frac{\sum{y_i}}{n}\right)^2 $$

### Sliding Windows for Regression

In [28]:
def best_var_window(df: pd.DataFrame) -> tuple[Optional[float], float]:

    # Initialization of variables
    best_variance = float('inf')
    best_split_value = None
    
    # Step 1: Sort the dataframe by the 'studied' column
    sorted_df = df.sort_values(by='studied')
    x = sorted_df['studied'].values
    y = sorted_df['score'].values
    n = len(y)

    # Step 2: Initialize LEFT Node and RIGHT Node counts
    left_count = 0
    left_sum = 0
    left_y_square_sum = 0

    right_count = n
    right_sum = y.sum()
    right_y_square_sum = np.sum(y**2) 

    for i in range(len(x) - 1):
        mid_point = (x[i] + x[i+1]) / 2
        left_sum += y[i]
        right_sum -= y[i]

        left_count += 1
        right_count -= 1
        n_total = left_count + right_count

        var_left = (left_y_square_sum / left_count) - (left_sum/left_count)**2
        var_right = (right_y_square_sum / right_count) - (right_sum/right_count)**2

        #print(f"Variance for left split: {var_left:.2f}")
        #print(f"Variance for right split: {var_right:.2f}")

        current_weighted_variance = (left_count / n_total) * var_left + (right_count / n_total) * var_right
        print(f"variance: {current_weighted_variance:.2f}, threshold: {mid_point}")
        
        if current_weighted_variance < best_variance:
            best_variance = current_weighted_variance
            best_split_value = mid_point
            
    return best_split_value, best_variance

In [29]:
best_split_value, best_variance = best_var_window(df)
print(f"Best split value: {best_split_value}, Best variance: {best_variance:.2f}")

variance: 552.91, threshold: 1.5
variance: 375.73, threshold: 2.5
variance: 245.91, threshold: 3.5
variance: 199.55, threshold: 4.5
variance: 160.81, threshold: 5.5
variance: 131.79, threshold: 6.5
variance: 212.62, threshold: 7.5
variance: 297.31, threshold: 8.5
variance: 360.20, threshold: 9.5
variance: 458.78, threshold: 10.5
variance: 562.08, threshold: 11.5
variance: 639.53, threshold: 12.5
variance: 693.98, threshold: 13.5
variance: 743.83, threshold: 14.5
Best split value: 6.5, Best variance: 131.79


## Optimization 2: Cumulative Sum Algorithm (CUMSUM)

**The next optimization technique is cumulative sum. Instead of adding 1 and removing 1 according to the labels, we compute all state of positive and negative label.** 

The following function just need to display the number of positive label and negative label, as we slide across the window. We need this info to confirm our cumulative sum computation.

In [17]:
def show_pos_neg_in_windows(df: pd.DataFrame):

    # Step 1: Sort the dataframe by the 'studied' column
    sorted_df = df.sort_values(by='studied')
    y = sorted_df['passed'].values
    n = len(y)

    # Step 2: Initialize LEFT Node counts
    left_total = 0
    left_positive = 0
    left_negative = 0

    right_total = n
    right_positive = (y == 1).sum()
    right_negative = (y == 0).sum()

    # Step 3: Move one sample from RIGHT to LEFT
    # Step 4: Update counts
    for i in range(n-1):
        left_total += 1
        right_total -= 1
        if y[i] == 1:
            left_positive += 1
            right_positive -= 1
        else:
            left_negative += 1
            right_negative -= 1
        
        print(f"Position {i}")
        print(f"Left Negative: {left_negative}")
        print(f"Left Positive: {left_positive}")
        print(f"Left Total: {left_total}")
    
        print(f"Right Negative: {right_negative}")
        print(f"Right Positive: {right_positive}")
        print(f"Right Total: {right_total}")
        print('---')


In [18]:
show_pos_neg_in_windows(df)

Position 0
Left Negative: 1
Left Positive: 0
Left Total: 1
Right Negative: 3
Right Positive: 8
Right Total: 11
---
Position 1
Left Negative: 2
Left Positive: 0
Left Total: 2
Right Negative: 2
Right Positive: 8
Right Total: 10
---
Position 2
Left Negative: 3
Left Positive: 0
Left Total: 3
Right Negative: 1
Right Positive: 8
Right Total: 9
---
Position 3
Left Negative: 4
Left Positive: 0
Left Total: 4
Right Negative: 0
Right Positive: 8
Right Total: 8
---
Position 4
Left Negative: 4
Left Positive: 1
Left Total: 5
Right Negative: 0
Right Positive: 7
Right Total: 7
---
Position 5
Left Negative: 4
Left Positive: 2
Left Total: 6
Right Negative: 0
Right Positive: 6
Right Total: 6
---
Position 6
Left Negative: 4
Left Positive: 3
Left Total: 7
Right Negative: 0
Right Positive: 5
Right Total: 5
---
Position 7
Left Negative: 4
Left Positive: 4
Left Total: 8
Right Negative: 0
Right Positive: 4
Right Total: 4
---
Position 8
Left Negative: 4
Left Positive: 5
Left Total: 9
Right Negative: 0
Right Pos

In the following, we use cumulative sum for left node and compare the sliding windows.

In [19]:
y = df['passed'].values

In [20]:
neg_left = (y == 0).cumsum()
pos_left = (y == 1).cumsum()
print('Negative Left', neg_left)
print('Positive Left', pos_left)

Negative Left [1 2 3 4 4 4 4 4 4 4 4 4]
Positive Left [0 0 0 0 1 2 3 4 5 6 7 8]


In [21]:
neg_total = neg_left[-1]
pos_total = pos_left[-1]
print('Negative Total', neg_total)
print('Positive Total', pos_total)

Negative Total 4
Positive Total 8


In [22]:
neg_right = neg_total - neg_left
pos_right = pos_total - pos_left
print('Negative Right', neg_right)
print('Positive Right', pos_right)

Negative Right [3 2 1 0 0 0 0 0 0 0 0 0]
Positive Right [8 8 8 8 7 6 5 4 3 2 1 0]


In [23]:
def gini_search_cumsum(df: pd.DataFrame) -> tuple[Optional[float], float]:
    
    # initialize best_gini values
    best_gini = float('inf')
    best_split_value = None

    # Step 1: Sort the dataframe by the 'studied' column
    sorted_df = df.sort_values(by='studied')
    x = sorted_df['studied'].values
    y = sorted_df['passed'].values
    n = len(y)

    # Step 2: Initialize LEFT and RIGHT Node cumulative sum
    negative_left = (y == 0).cumsum()
    positive_left = (y == 1).cumsum()
    negative_total = negative_left[-1]
    positive_total = positive_left[-1]
    negative_right = negative_total - negative_left
    positive_right = positive_total - positive_left
    left_total = 0
    right_total = n

    # Step 3: Move one sample from RIGHT to LEFT
    # Step 4: Update counts
    for i in range(n-1):
        
        left_total += 1
        right_total -= 1
  

        # Compute weighted gini
        left_gini = gini_v2(negative_left[i], positive_left[i])
        right_gini = gini_v2(negative_right[i], positive_right[i])
        weighted_gini = ((left_total/n) * left_gini) + ((right_total/n) * right_gini)
        print(f"Weighted Gini: {weighted_gini:.2f}")

        if weighted_gini < best_gini:
            best_gini = weighted_gini
            if x[i] == x[i+1]:
                continue
            else:
                best_split_value = (x[i] + x[i+1]) / 2
            print("Midpoint Threshold", best_split_value)
        
        print('---')
    
    return best_split_value, best_gini

In [24]:
best_split_value, best_gini = gini_search_cumsum(df)
print(f"Best split value: {best_split_value}, Best Gini impurity: {best_gini:.2f}")

Weighted Gini: 0.36
Midpoint Threshold 1.5
---
Weighted Gini: 0.27
Midpoint Threshold 2.5
---
Weighted Gini: 0.15
Midpoint Threshold 3.5
---
Weighted Gini: 0.00
Midpoint Threshold 4.5
---
Weighted Gini: 0.13
---
Weighted Gini: 0.22
---
Weighted Gini: 0.29
---
Weighted Gini: 0.33
---
Weighted Gini: 0.37
---
Weighted Gini: 0.40
---
Weighted Gini: 0.42
---
Best split value: 4.5, Best Gini impurity: 0.00


## Optimization 3: Vectorization

**In the previous code, we still need to loop through every element in the feature. In this optimization, we compute the weighted gini in vectorized form.**

In [25]:
neg_left = (y == 0).cumsum()
pos_left = (y == 1).cumsum()
print('Negative Left', neg_left)
print('Positive Left', pos_left)

Negative Left [1 2 3 4 4 4 4 4 4 4 4 4]
Positive Left [0 0 0 0 1 2 3 4 5 6 7 8]


In [26]:
neg_total = neg_left[-1]
pos_total = pos_left[-1]
print('Negative Total', neg_total)
print('Positive Total', pos_total)

Negative Total 4
Positive Total 8


In [27]:
neg_right = neg_total - neg_left
pos_right = pos_total - pos_left
print('Negative Right', neg_right)
print('Positive Right', pos_right)

Negative Right [3 2 1 0 0 0 0 0 0 0 0 0]
Positive Right [8 8 8 8 7 6 5 4 3 2 1 0]


Since we are not looping through `n-1`, we need to remove the last element in the list for the cumulative sum.

In [28]:
neg_left = neg_left[:-1]
pos_left = pos_left[:-1]
print('Negative Left', neg_left)
print('Positive Left', pos_left)

Negative Left [1 2 3 4 4 4 4 4 4 4 4]
Positive Left [0 0 0 0 1 2 3 4 5 6 7]


In [29]:
neg_right = neg_right[:-1]
pos_right = pos_right[:-1]
print('Negative Right', neg_right)
print('Positive Right', pos_right)

Negative Right [3 2 1 0 0 0 0 0 0 0 0]
Positive Right [8 8 8 8 7 6 5 4 3 2 1]


In [30]:
left_count = neg_left + pos_left
left_count

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])

In [31]:
right_count = neg_right + pos_right
right_count

array([11, 10,  9,  8,  7,  6,  5,  4,  3,  2,  1])

Testing weighted gini computation on vectorized form. 

In [32]:
neg_left/left_count

array([1.        , 1.        , 1.        , 1.        , 0.8       ,
       0.66666667, 0.57142857, 0.5       , 0.44444444, 0.4       ,
       0.36363636])

In [33]:
pos_left/left_count

array([0.        , 0.        , 0.        , 0.        , 0.2       ,
       0.33333333, 0.42857143, 0.5       , 0.55555556, 0.6       ,
       0.63636364])

In [34]:
neg_right/right_count

array([0.27272727, 0.2       , 0.11111111, 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        ])

In [35]:
pos_right/right_count

array([0.72727273, 0.8       , 0.88888889, 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        ])

In [36]:
# Create gini for vector support
def gini_vec(negative: np.array, positive: np.array, count: np.array) -> float:
    '''
    Compute the Gini impurity for a set of labels.
    negative: refers to number of negative labels in vector
    positive: refers to number of positive labels in vector
    count: total number of labels in vector
    '''
    
    p0 = negative/count
    p1 = positive/count

    gini = 1 - (p0**2 + p1**2)
    return gini

In [37]:
def gini_search_near_CART(df: pd.DataFrame) -> tuple[Optional[float], float]:
    # initialize best_gini values
    best_gini = float('inf')
    best_split_value = None

    # Step 1: Sort the dataframe by the 'studied' column
    sorted_df = df.sort_values(by='studied')
    x = sorted_df['studied'].values
    y = sorted_df['passed'].values
    n = len(y)

    # Step 2: Compute cumulative sum in vector form
    negative_left = (y == 0).cumsum()
    positive_left = (y == 1).cumsum()

    negative_total = negative_left[-1]
    positive_total = positive_left[-1]

    negative_right = negative_total - negative_left
    positive_right = positive_total - positive_left

    # SHIFT EVERYTHING TO SPLIT SPACE (n-1)
    negative_left = negative_left[:-1]
    positive_left = positive_left[:-1]
    negative_right = negative_right[:-1]
    positive_right = positive_right[:-1]

    count_left = negative_left + positive_left
    count_right = negative_right + positive_right

    # Step 3: Compute Weighted Gini
    left_gini = gini_vec(negative_left, positive_left, count_left)
    right_gini = gini_vec(negative_right, positive_right, count_right)
    weighted_gini = ((count_left/n) * left_gini) + ((count_right/n) * right_gini)
    print(weighted_gini)

    # Step 4: Get best gini using argmin
    best_gini_index = weighted_gini.argmin()
    best_gini = weighted_gini.min()

    best_split_value = (x[best_gini_index] + x[best_gini_index + 1]) /2
    print(f"Best Gini : {best_gini}, Best Gini Index : {best_gini_index}, Best Threshold : {best_split_value}")
    
    return best_split_value, best_gini

In [38]:
best_split_value, best_gini = gini_search_near_CART(df)
print(f"Best split value: {best_split_value}, Best Gini impurity: {best_gini:.2f}")

[0.36363636 0.26666667 0.14814815 0.         0.13333333 0.22222222
 0.28571429 0.33333333 0.37037037 0.4        0.42424242]
Best Gini : 0.0, Best Gini Index : 3, Best Threshold : 4.5
Best split value: 4.5, Best Gini impurity: 0.00


## Problem with Repeated Value

For feature values that has repetition, it would seems a waste in computing resource if we computed the weighted Gini for repeated values. This will be extremely obviously of we are dealing to millions of rows and we have repeated feature values for 50% of them. Thus the most optimal way of to only compute the weighted Gini on the boundaries.  

In [39]:
df2 = pd.DataFrame({'studied': [1,1,1,2,2,2,2,3,3,3,4,4], 
        'passed': [0,0,0,0,1,1,1,1,1,1,1,1]})

In [40]:
best_split_value, best_gini = gini_search_near_CART(df2)
print(f"Best split value: {best_split_value}, Best Gini impurity: {best_gini:.2f}")

[0.36363636 0.26666667 0.14814815 0.         0.13333333 0.22222222
 0.28571429 0.33333333 0.37037037 0.4        0.42424242]
Best Gini : 0.0, Best Gini Index : 3, Best Threshold : 2.0
Best split value: 2.0, Best Gini impurity: 0.00


We also have another problem here where feature values of 3 have a mixed of labels. Using previous method, we could not get the best threshold.

In [41]:
x = df2['studied'].values
y = df2['passed'].values
n = len(y)

In [42]:
valid = x[:-1] != x[1:]
valid

array([False, False,  True, False, False, False,  True, False, False,
        True, False])

In [43]:
# Cumulative sum for left node
left_negative = (y == 0).cumsum()
left_positive = (y == 1).cumsum()
print('Left Negative', left_negative)
print('Left Positive', left_positive)

Left Negative [1 2 3 4 4 4 4 4 4 4 4 4]
Left Positive [0 0 0 0 1 2 3 4 5 6 7 8]


In [44]:
negative_total = left_negative[-1]
positive_total = left_positive[-1]
print('Negative Total', negative_total)
print('Positive Total', positive_total)

Negative Total 4
Positive Total 8


In [45]:
# Cumulative sum for left node
right_negative = negative_total - left_negative
right_positive = positive_total - left_positive
print('Right Negative', right_negative)
print('Right Positive', right_positive)

Right Negative [3 2 1 0 0 0 0 0 0 0 0 0]
Right Positive [8 8 8 8 7 6 5 4 3 2 1 0]


In [46]:
left_negative = left_negative[:-1]
left_positive = left_positive[:-1]
left_count = left_negative + left_positive

print('Left Negative', left_negative)
print('Left Positive', left_positive)
print('Left total', left_count)

Left Negative [1 2 3 4 4 4 4 4 4 4 4]
Left Positive [0 0 0 0 1 2 3 4 5 6 7]
Left total [ 1  2  3  4  5  6  7  8  9 10 11]


In [47]:
right_negative = right_negative[:-1]
right_positive = right_positive[:-1]
right_count = right_negative + right_positive

print('Right Negative', left_negative)
print('Right Positive', left_positive)
print('Right total', left_count)

Right Negative [1 2 3 4 4 4 4 4 4 4 4]
Right Positive [0 0 0 0 1 2 3 4 5 6 7]
Right total [ 1  2  3  4  5  6  7  8  9 10 11]


In [48]:
left_negative = left_negative[valid]
left_positive = left_positive[valid]

print('Left Negative', left_negative)
print('Left Positive', left_positive)

Left Negative [3 4 4]
Left Positive [0 3 6]


In [49]:
right_negative = right_negative[valid]
right_positive = right_positive[valid]

print('Right Negative', right_negative)
print('Right Positive', right_positive)

Right Negative [1 0 0]
Right Positive [8 5 2]


In [50]:
left_count = left_negative + left_positive
right_count = right_negative + right_positive

print("Left count", left_count)
print("Right count", right_count)

Left count [ 3  7 10]
Right count [9 5 2]


In [51]:
left_gini = gini_vec(left_negative, left_positive, left_count)
right_gini = gini_vec(right_negative, right_positive, right_count)
weighted_gini = (left_count / n) *  left_gini + (right_count / n) * right_gini
weighted_gini

array([0.14814815, 0.28571429, 0.4       ])

In [52]:

def best_gini_CART_ready(df:  pd.DataFrame):

    best_gini = float('inf')
    best_split_value = None

    # Sort features
    sorted_df = df.sort_values(by='studied')

    # Convert to numpy
    x = sorted_df['studied'].values
    y = sorted_df['passed'].values

    # Compute boundaries
    valid = x[:-1] != x[1:]

    # Perform cumsum
    left_negative = (y == 0).cumsum()
    left_positive = (y == 1).cumsum()
    negative_total = left_negative[-1]
    positive_total = left_positive[-1]
    right_negative = negative_total - left_negative
    right_positive = positive_total - left_positive

    # Remove last element
    left_negative = left_negative[:-1]
    left_positive = left_positive[:-1]
    right_negative = right_negative[:-1]
    right_positive = right_positive[:-1]

    # Initialize total number of data
    n = len(y)

    # Get the boundaries
    left_negative = left_negative[valid]
    left_positive = left_positive[valid]
    right_negative = right_negative[valid]
    right_positive = right_positive[valid]

    # Get the total of left and right nodes
    left_count = left_negative + left_positive
    right_count = right_negative + right_positive
    
    # Compute gini for left and right nodes
    left_gini = gini_vec(left_negative, left_positive, left_count)
    right_gini = gini_vec(right_negative, right_positive, right_count)
    
    # Compute weighted gini
    weighted_gini = (left_count / n) *  left_gini + (right_count / n) * right_gini
    print('weighted gini',weighted_gini)

    # Get the best gini using min and argmin
    best_gini = weighted_gini.min()
    best_gini_index = weighted_gini.argmin()
    # Compute midpoint for the lowest weighted gini
    reduced_x = (x[:-1][valid] + x[1:][valid]) / 2

    best_split_value = reduced_x[best_gini_index]

    return best_split_value, best_gini

In [53]:
best_split_value, best_gini = best_gini_CART_ready(df2)
print(f"Best split value: {best_split_value}, Best Gini impurity: {best_gini:.4f}")

weighted gini [0.14814815 0.28571429 0.4       ]
Best split value: 1.5, Best Gini impurity: 0.1481


In [71]:
def built_tree(df):

    if df.empty:
        return None

    # If there's only one row left, return its label
    if len(df) <= 1:
        print(f"Single row subset found with label: {int(df['passed'].iloc[0])}")
        return int(df['passed'].iloc[0])
    
    # If all labels are identical, return that label
    # nunique() returns the number of unique values in the 'passed' column
    if df['passed'].nunique() == 1: 
        print(f"Pure subset found with label: {int(df['passed'].iloc[0])}")
        return int(df['passed'].iloc[0])


    # Find the best split based on Gini impurity
    threshold_studied, best_gini = best_gini_CART_ready(df)
    print(f"Best split value: {threshold_studied}, Gini after split: {best_gini:.2f}")

    # Split the dataset into two subsets based on the threshold value of 'studied'
    left_subset = df[df['studied'] <= threshold_studied]
    right_subset = df[df['studied'] > threshold_studied]

    print(f"Left subset:\n{left_subset}\n")
    print(f"Right subset:\n{right_subset}\n")

    # Create a dictionary to represent the current node
    clf = {'threshold_studied': float(threshold_studied)}

    # Recursively build the left and right subtrees
    clf['left'] = built_tree(left_subset)
    clf['right'] = built_tree(right_subset)

    return clf

In [70]:
clf = built_tree(df2)

weighted gini [0.14814815 0.28571429 0.4       ]
Best split value: 1.5, Gini after split: 0.15
Left subset:
   studied  passed
0        1       0
1        1       0
2        1       0

Right subset:
    studied  passed
3         2       0
4         2       1
5         2       1
6         2       1
7         3       1
8         3       1
9         3       1
10        4       1
11        4       1

Pure subset found with label: 0
weighted gini [0.16666667 0.19047619]
Best split value: 2.5, Gini after split: 0.17
Left subset:
   studied  passed
3        2       0
4        2       1
5        2       1
6        2       1

Right subset:
    studied  passed
7         3       1
8         3       1
9         3       1
10        4       1
11        4       1

weighted gini []


ValueError: zero-size array to reduction operation minimum which has no identity

There are error occurs in the previous code, this is because in df2 there is mixed with positive and negative labels. Since we are using boundaries, this error occurs when there is no more valid split available.  

<details>

<summary> LLM Error Analysis </summary>

I think I know what's happening. The code itself looks correct, but there's **one important edge case** that recursive tree building exposes.

### The likely problem

Consider a subset like:

```text
studied  passed
-------  ------
2        0
2        1
2        1
```

The labels are mixed, so this check fails:

```python
if df['passed'].nunique() == 1:
    ...
```

But all feature values are identical:

```python
x = [2, 2, 2]
```

Then:

```python
valid = x[:-1] != x[1:]
```

becomes

```python
[False, False]
```

There are **no valid CART split positions**.

---

### What happens next?

After filtering:

```python
left_negative = left_negative[valid]
```

everything becomes empty:

```python
left_negative = []
left_positive = []
...
weighted_gini = []
```

Then this line crashes:

```python
best_gini = weighted_gini.min()
```

because NumPy cannot compute the minimum of an empty array.

Typical errors are:

```text
ValueError: zero-size array to reduction operation minimum which has no identity
```

or later:

```text
IndexError
```

---

## This is a normal CART stopping condition

A decision tree stops not only when the labels are pure.

It also stops when **no further split is possible**.

The common stopping conditions are:

```text
1. Node is empty
2. Node has one sample
3. All labels are identical
4. No valid split exists   ← you're missing this
5. max_depth reached (optional)
6. min_samples_split reached (optional)
```

---

## Fix `best_gini_CART_ready`

Right after computing `valid`:

```python
valid = x[:-1] != x[1:]

if not valid.any():
    return None, float("inf")
```

---

## Then fix `built_tree`

After:

```python
threshold_studied, best_gini = best_gini_CART_ready(df)
```

check whether a split exists:

```python
if threshold_studied is None:
    # No valid split exists.
    # Return the majority class.
    return int(df["passed"].mode()[0])
```

This is exactly what CART implementations do.

---

## Why return the majority class?

Imagine this node:

| studied | passed |
| ------: | -----: |
|       2 |      0 |
|       2 |      1 |
|       2 |      1 |

There is no threshold that can separate the labels because every sample has the same feature value.

The best the tree can do is predict the majority class:

```text
0 : 1 sample
1 : 2 samples

Prediction = 1
```

---

## I would also simplify your stopping logic

Instead of:

```python
if len(df) <= 1:
    ...

if df['passed'].nunique() == 1:
    ...
```

I'd write:

```python
# Empty node
if df.empty:
    return None

# Pure node
if df["passed"].nunique() == 1:
    return int(df["passed"].iloc[0])

# No possible split
threshold, gini = best_gini_CART_ready(df)

if threshold is None:
    return int(df["passed"].mode()[0])
```

The `len(df) <= 1` case is already covered:

* If there's one sample, the labels are necessarily pure (`nunique() == 1`).

---

In [80]:
def best_gini_CART_ready(df:  pd.DataFrame):

    best_gini = float('inf')
    best_split_value = None

    # Sort features
    sorted_df = df.sort_values(by='studied')

    # Convert to numpy
    x = sorted_df['studied'].values
    y = sorted_df['passed'].values

    # Compute boundaries
    valid = x[:-1] != x[1:]

    # No valid CART split exists
    if not valid.any():
        return None, float("inf")

    # Perform cumsum
    left_negative = (y == 0).cumsum()
    left_positive = (y == 1).cumsum()
    negative_total = left_negative[-1]
    positive_total = left_positive[-1]
    right_negative = negative_total - left_negative
    right_positive = positive_total - left_positive

    # Remove last element
    left_negative = left_negative[:-1]
    left_positive = left_positive[:-1]
    right_negative = right_negative[:-1]
    right_positive = right_positive[:-1]

    # Initialize total number of data
    n = len(y)

    # Get the boundaries
    left_negative = left_negative[valid]
    left_positive = left_positive[valid]
    right_negative = right_negative[valid]
    right_positive = right_positive[valid]

    # Get the total of left and right nodes
    left_count = left_negative + left_positive
    right_count = right_negative + right_positive
    
    # Compute gini for left and right nodes
    left_gini = gini_vec(left_negative, left_positive, left_count)
    right_gini = gini_vec(right_negative, right_positive, right_count)
    
    # Compute weighted gini
    weighted_gini = (left_count / n) *  left_gini + (right_count / n) * right_gini
    print('weighted gini',weighted_gini)

    # Get the best gini using min and argmin
    best_gini = weighted_gini.min()
    best_gini_index = weighted_gini.argmin()
    # Compute midpoint for the lowest weighted gini
    reduced_x = (x[:-1][valid] + x[1:][valid]) / 2

    best_split_value = reduced_x[best_gini_index]

    return best_split_value, best_gini

In [ ]:
def built_tree(df):

    if df.empty:
        return None

    # If there's only one row left, return its label
    if len(df) <= 1:
        print(f"Single row subset found with label: {int(df['passed'].iloc[0])}")
        return int(df['passed'].iloc[0])
    
    # If all labels are identical, return that label
    # nunique() returns the number of unique values in the 'passed' column
    if df['passed'].nunique() == 1: 
        print(f"Pure subset found with label: {int(df['passed'].iloc[0])}")
        return int(df['passed'].iloc[0])
    
    # Find the best split based on Gini impurity
    threshold_studied, best_gini = best_gini_CART_ready(df)
    print(f"Best split value: {threshold_studied}, Gini after split: {best_gini:.2f}")

    # Another stopping condition to catch invalid threshold
    if threshold_studied is None:
        return int(df["passed"].mode()[0])

    # Split the dataset into two subsets based on the threshold value of 'studied'
    left_subset = df[df['studied'] <= threshold_studied]
    right_subset = df[df['studied'] > threshold_studied]

    print(f"Left subset:\n{left_subset}\n")
    print(f"Right subset:\n{right_subset}\n")

    # Create a dictionary to represent the current node
    clf = {'threshold_studied': float(threshold_studied)}

    # Recursively build the left and right subtrees
    clf['left'] = built_tree(left_subset)
    clf['right'] = built_tree(right_subset)

    return clf

In [77]:
clf = built_tree(df2)

weighted gini [0.14814815 0.28571429 0.4       ]
Best split value: 1.5, Gini after split: 0.15
Left subset:
   studied  passed
0        1       0
1        1       0
2        1       0

Right subset:
    studied  passed
3         2       0
4         2       1
5         2       1
6         2       1
7         3       1
8         3       1
9         3       1
10        4       1
11        4       1

Pure subset found with label: 0
weighted gini [0.16666667 0.19047619]
Best split value: 2.5, Gini after split: 0.17
Left subset:
   studied  passed
3        2       0
4        2       1
5        2       1
6        2       1

Right subset:
    studied  passed
7         3       1
8         3       1
9         3       1
10        4       1
11        4       1

Best split value: None, Gini after split: inf
Pure subset found with label: 1


In [78]:
clf

{'threshold_studied': 1.5,
 'left': 0,
 'right': {'threshold_studied': 2.5, 'left': 1, 'right': 1}}

### Create Decision Tree Function

In [79]:
# Create gini for vector support
def gini_vec(negative: np.array, positive: np.array, count: np.array) -> float:
    '''
    Compute the Gini impurity for a set of labels.
    negative: refers to number of negative labels in vector
    positive: refers to number of positive labels in vector
    count: total number of labels in vector
    '''
    
    p0 = negative/count
    p1 = positive/count

    gini = 1 - (p0**2 + p1**2)
    return gini

In [81]:
def best_gini_CART_ready(df:  pd.DataFrame):

    best_gini = float('inf')
    best_split_value = None

    # Sort features
    sorted_df = df.sort_values(by='studied')

    # Convert to numpy
    x = sorted_df['studied'].values
    y = sorted_df['passed'].values

    # Compute boundaries
    valid = x[:-1] != x[1:]

    # No valid CART split exists
    if not valid.any():
        return None, float("inf")

    # Perform cumsum
    left_negative = (y == 0).cumsum()
    left_positive = (y == 1).cumsum()
    negative_total = left_negative[-1]
    positive_total = left_positive[-1]
    right_negative = negative_total - left_negative
    right_positive = positive_total - left_positive

    # Remove last element
    left_negative = left_negative[:-1]
    left_positive = left_positive[:-1]
    right_negative = right_negative[:-1]
    right_positive = right_positive[:-1]

    # Initialize total number of data
    n = len(y)

    # Get the boundaries
    left_negative = left_negative[valid]
    left_positive = left_positive[valid]
    right_negative = right_negative[valid]
    right_positive = right_positive[valid]

    # Get the total of left and right nodes
    left_count = left_negative + left_positive
    right_count = right_negative + right_positive
    
    # Compute gini for left and right nodes
    left_gini = gini_vec(left_negative, left_positive, left_count)
    right_gini = gini_vec(right_negative, right_positive, right_count)
    
    # Compute weighted gini
    weighted_gini = (left_count / n) *  left_gini + (right_count / n) * right_gini
    print('weighted gini',weighted_gini)

    # Get the best gini using min and argmin
    best_gini = weighted_gini.min()
    best_gini_index = weighted_gini.argmin()
    # Compute midpoint for the lowest weighted gini
    reduced_x = (x[:-1][valid] + x[1:][valid]) / 2

    best_split_value = reduced_x[best_gini_index]

    return best_split_value, best_gini

In [82]:
def built_tree(df):

    if df.empty:
        return None

    # If there's only one row left, return its label
    if len(df) <= 1:
        print(f"Single row subset found with label: {int(df['passed'].iloc[0])}")
        return int(df['passed'].iloc[0])
    
    # If all labels are identical, return that label
    # nunique() returns the number of unique values in the 'passed' column
    if df['passed'].nunique() == 1: 
        print(f"Pure subset found with label: {int(df['passed'].iloc[0])}")
        return int(df['passed'].iloc[0])
    
    # Find the best split based on Gini impurity
    threshold_studied, best_gini = best_gini_CART_ready(df)
    print(f"Best split value: {threshold_studied}, Gini after split: {best_gini:.2f}")

    # Another stopping condition to catch invalid threshold
    if threshold_studied is None:
        return int(df["passed"].mode()[0])

    # Split the dataset into two subsets based on the threshold value of 'studied'
    left_subset = df[df['studied'] <= threshold_studied]
    right_subset = df[df['studied'] > threshold_studied]

    print(f"Left subset:\n{left_subset}\n")
    print(f"Right subset:\n{right_subset}\n")

    # Create a dictionary to represent the current node
    clf = {'threshold_studied': float(threshold_studied)}

    # Recursively build the left and right subtrees
    clf['left'] = built_tree(left_subset)
    clf['right'] = built_tree(right_subset)

    return clf

### Build Tree 

We build a tree using current data frame.

In [83]:
df

,studied,passed
0,1,0
1,2,0
2,3,0
3,4,0
4,5,1
5,6,1
6,7,1
7,8,1
8,9,1
9,10,1


In [84]:
clf = built_tree(df)

weighted gini [0.36363636 0.26666667 0.14814815 0.         0.13333333 0.22222222
 0.28571429 0.33333333 0.37037037 0.4        0.42424242]
Best split value: 4.5, Gini after split: 0.00
Left subset:
   studied  passed
0        1       0
1        2       0
2        3       0
3        4       0

Right subset:
    studied  passed
4         5       1
5         6       1
6         7       1
7         8       1
8         9       1
9        10       1
10       11       1
11       12       1

Pure subset found with label: 0
Pure subset found with label: 1


In [85]:
clf


{'threshold_studied': 4.5, 'left': 0, 'right': 1}

In [86]:
# print tree structure in json format
print(json.dumps(clf, indent=8))


{
        "threshold_studied": 4.5,
        "left": 0,
        "right": 1
}


In [87]:
# print tree structure with guided lines
def print_tree(node, indent="    "):
    if isinstance(node, dict):
        print(f"{indent}Threshold studied: {node['threshold_studied']}")
        print(f"{indent}├─ Left Node: <={node['threshold_studied']}")
        print_tree(node['left'], indent + "│   ")
        print(f"{indent}└─ Right Node: >{node['threshold_studied']}")
        print_tree(node['right'], indent + "    ")
    else:
        print(f"{indent}Passed: {node}")


In [88]:
print_tree(clf)

    Threshold studied: 4.5
    ├─ Left Node: <=4.5
    │   Passed: 0
    └─ Right Node: >4.5
        Passed: 1


### Prediction / Inference

To make a prediction we pass the prediction to the tree until we reach a leaf.

In [89]:
# make prediction function
def predict(clf, x):
    node = clf  # start at root of the tree

    # while we are still at a decision node (dictionary)
    while isinstance(node, dict):

        # get the split threshold stored at this node
        threshold = node['threshold_studied']

        # go left if x is smaller or equal to threshold
        if x <= threshold:
            print(f"Going left: {x} <= {threshold}")
            node = node['left']
        # otherwise go right
        else:
            print(f"Going right: {x} > {threshold}")
            node = node['right']

    # when we reach a leaf (not a dict), return the prediction
    return node

In [90]:
number_of_hours_studied = 5
prediction = predict(clf, number_of_hours_studied)
print(f"Prediction for student who studied {number_of_hours_studied} hours: {prediction}")

Going right: 5 > 4.5
Prediction for student who studied 5 hours: 1


## Scikit-Learn Compare

In the following, we will use Scikit-Learn to compare the tree. 

In [91]:
from sklearn.tree import DecisionTreeClassifier

sklearn_clf = DecisionTreeClassifier(criterion='gini')

sklearn_clf.fit(df[['studied']], df['passed'])

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [92]:
# get tree structure in json format
from sklearn.tree import export_text
tree_rules = export_text(sklearn_clf, feature_names=['studied'])
print(tree_rules)

|--- studied <= 4.50
|   |--- class: 0
|--- studied >  4.50
|   |--- class: 1



In [93]:
# make prediction function using sklearn
df_numbers_of_hours_studied = pd.DataFrame({'studied': [number_of_hours_studied]}) 
# need to convert to df as original data during training is in df format

sklearn_prediction = sklearn_clf.predict(df_numbers_of_hours_studied)
print(f"Prediction for student who studied {number_of_hours_studied} hours: {sklearn_prediction}")

Prediction for student who studied 5 hours: [1]


## DF2

In [94]:
df2

,studied,passed
0,1,0
1,1,0
2,1,0
3,2,0
4,2,1
5,2,1
6,2,1
7,3,1
8,3,1
9,3,1


In [95]:
clf = built_tree(df2)

weighted gini [0.14814815 0.28571429 0.4       ]
Best split value: 1.5, Gini after split: 0.15
Left subset:
   studied  passed
0        1       0
1        1       0
2        1       0

Right subset:
    studied  passed
3         2       0
4         2       1
5         2       1
6         2       1
7         3       1
8         3       1
9         3       1
10        4       1
11        4       1

Pure subset found with label: 0
weighted gini [0.16666667 0.19047619]
Best split value: 2.5, Gini after split: 0.17
Left subset:
   studied  passed
3        2       0
4        2       1
5        2       1
6        2       1

Right subset:
    studied  passed
7         3       1
8         3       1
9         3       1
10        4       1
11        4       1

Best split value: None, Gini after split: inf
Pure subset found with label: 1


In [96]:
clf

{'threshold_studied': 1.5,
 'left': 0,
 'right': {'threshold_studied': 2.5, 'left': 1, 'right': 1}}

In [97]:
print_tree(clf)

    Threshold studied: 1.5
    ├─ Left Node: <=1.5
    │   Passed: 0
    └─ Right Node: >1.5
        Threshold studied: 2.5
        ├─ Left Node: <=2.5
        │   Passed: 1
        └─ Right Node: >2.5
            Passed: 1


In [98]:
sklearn_clf = DecisionTreeClassifier(criterion='gini')

sklearn_clf.fit(df2[['studied']], df2['passed'])

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [99]:
# get tree structure in json format
from sklearn.tree import export_text
tree_rules = export_text(sklearn_clf, feature_names=['studied'])
print(tree_rules)

|--- studied <= 1.50
|   |--- class: 0
|--- studied >  1.50
|   |--- studied <= 2.50
|   |   |--- class: 1
|   |--- studied >  2.50
|   |   |--- class: 1



## End